# LeetCode 1910 — Remove All Occurrences of a Substring bằng Python

Link đề: https://leetcode.com/problems/remove-all-occurrences-of-a-substring/ · Độ khó: **Medium** · Chủ đề: String, Stack, Simulation (String Matching / KMP)

Skeleton LeetCode:

```python
class Solution:
    def removeOccurrences(self, s: str, part: str) -> str:
        ...
```

Lộ trình: **mô phỏng đúng câu chữ** (tìm lần xuất hiện trái nhất bằng vòng lặp tay, cắt bỏ) → **`str.replace(part, "", 1)` lặp** → **ngăn xếp + so hậu tố** `O(n·m)` → **ngăn xếp + KMP (hàm tiền tố)** `O(n + m)`. Kèm chứng minh vì sao ngăn xếp cho **đúng** kết quả của "luôn xóa lần xuất hiện trái nhất".

## 1. Mục tiêu học

1. Mô phỏng chính xác thao tác "xóa lần xuất hiện **trái nhất**" và hiểu vì sao thứ tự xóa **quan trọng** khi các lần xuất hiện **chồng lấn** (ví dụ `part = "aba"`).
2. Dùng **ngăn xếp ký tự**: đẩy từng ký tự, hễ đỉnh ngăn xếp kết thúc bằng `part` thì bật ra — kỹ thuật "xóa lặp liên hoàn" giống bài khớp ngoặc.
3. Nắm **hàm tiền tố (prefix function) của KMP** và cách lưu **trạng thái khớp theo từng vị trí ngăn xếp** để khi bật ra có thể "quay về" trạng thái cũ trong `O(1)`.
4. Chứng minh tính đúng bằng nhận xét: hai lần xuất hiện cùng độ dài thì **bắt đầu sớm hơn ⇔ kết thúc sớm hơn**.
5. Kiểm chứng bằng vét cạn trên mọi chuỗi nhỏ, và thấy phản ví dụ cho `replace` toàn bộ / xóa lần xuất hiện phải nhất.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `part`. Lặp lại thao tác sau cho tới khi `s` **không còn** chứa `part`:
- tìm lần xuất hiện **trái nhất** của chuỗi con `part` trong `s` và xóa nó.

Trả về `s` sau khi xóa hết. (Chuỗi con là dãy ký tự **liên tiếp**.)

**Ví dụ 1:** `s = "daabcbaabcbc"`, `part = "abc"` → `"dab"`:
`"daabcbaabcbc"` → xóa tại 2 → `"dabaabcbc"` → xóa tại 4 → `"dababc"` → xóa tại 3 → `"dab"`.

**Ví dụ 2:** `s = "axxxxyyyyb"`, `part = "xy"` → `"ab"` (mỗi lần xóa `"xy"` ở giữa lại tạo ra `"xy"` mới).

**Ràng buộc:** `1 ≤ s.length ≤ 1000`, `1 ≤ part.length ≤ 1000`, chỉ gồm chữ thường.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [("daabcbaabcbc", "abc", "dab"), ("axxxxyyyyb", "xy", "ab")]
for s, p, want in examples:
    print(f"s={s!r}, part={p!r} -> kỳ vọng {want!r}")

s='daabcbaabcbc', part='abc' -> kỳ vọng 'dab'
s='axxxxyyyyb', part='xy' -> kỳ vọng 'ab'


## 3. Kiến thức nền

### 3.1 `str.find` và cắt chuỗi

`s.find(part)` trả chỉ số bắt đầu **trái nhất** (hoặc `-1`). Xóa: `s[:i] + s[i + len(part):]` — tạo chuỗi mới `O(len(s))`.

In [2]:
s, p = "daabcbaabcbc", "abc"
i = s.find(p)
print(i, "->", s[:i] + s[i + len(p):])

2 -> dabaabcbc


### 3.2 `str.replace(old, new, count)`

`count = 1` chỉ thay lần xuất hiện **đầu tiên** (trái nhất). Không có `count` thì thay **mọi** lần xuất hiện **không chồng lấn** quét từ trái — **khác** với đề (xem Lỗi 1).

In [3]:
print("daabcbaabcbc".replace("abc", "", 1))
print("daabcbaabcbc".replace("abc", ""))

dabaabcbc
dababc


### 3.3 Ngăn xếp bằng list và so sánh hậu tố

`st[-m:]` là `m` phần tử cuối. So với `list(part)` để biết đỉnh ngăn xếp có kết thúc bằng `part`; `del st[-m:]` bật `m` phần tử.

In [4]:
st = list("dab")
part = "ab"
print(st[-len(part):] == list(part))
del st[-len(part):]
print(st)

True
['d']


### 3.4 Hàm tiền tố (prefix function) của KMP

`pi[i]` = độ dài **tiền tố thật sự dài nhất** của `part[:i+1]` đồng thời là **hậu tố** của nó. Khi so khớp gặp ký tự không khớp ở trạng thái `j`, ta lùi `j = pi[j−1]` thay vì về 0.

In [5]:
def prefix_function(p):
    pi = [0] * len(p)
    for i in range(1, len(p)):
        j = pi[i - 1]
        while j and p[i] != p[j]:
            j = pi[j - 1]
        if p[i] == p[j]:
            j += 1
        pi[i] = j
    return pi

for p in ["abc", "aba", "aabaa", "abab"]:
    print(p, prefix_function(p))

abc [0, 0, 0]
aba [0, 0, 1]
aabaa [0, 1, 0, 1, 2]
abab [0, 0, 1, 2]


### 3.5 Bước chuyển của máy KMP

Từ trạng thái `j` (đã khớp `j` ký tự đầu của `part`) đọc ký tự `c`: lùi theo `pi` tới khi `part[j] == c` hoặc `j = 0`, rồi nếu khớp thì `j += 1`. Khi `j == len(part)` là vừa thấy một lần xuất hiện.

In [6]:
def kmp_step(j, c, p, pi):
    while j and p[j] != c:
        j = pi[j - 1]
    if p[j] == c:
        j += 1
    return j

p = "aba"; pi = prefix_function(p); j = 0
for c in "ababa":
    j = kmp_step(j, c, p, pi)
    print(f"đọc {c!r} -> trạng thái {j}{'  (khớp trọn!)' if j == len(p) else ''}")
    if j == len(p):
        j = pi[j - 1]

đọc 'a' -> trạng thái 1
đọc 'b' -> trạng thái 2
đọc 'a' -> trạng thái 3  (khớp trọn!)
đọc 'b' -> trạng thái 2
đọc 'a' -> trạng thái 3  (khớp trọn!)


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay Ví dụ 2 (`part = "xy"`):

```
axxxxyyyyb   -> "xy" trái nhất ở chỉ số 4 -> axxxyyyb
axxxyyyb     -> chỉ số 3 -> axxyyb
axxyyb       -> chỉ số 2 -> axyb
axyb         -> chỉ số 1 -> ab
```

Mỗi lần xóa, phần **bên trái** chỗ xóa và phần **bên phải** dính lại, có thể **tạo ra lần xuất hiện mới** ngay tại chỗ nối. Nhận xét: phần bên trái chỗ xóa **không chứa** `part` (vì ta xóa lần trái nhất)… nên lần xuất hiện mới chỉ có thể nằm ở chỗ nối hoặc bên phải. Đây là gợi ý cho **ngăn xếp**: đọc từ trái sang phải, giữ phần "đã sạch" trong ngăn xếp, hễ đỉnh ngăn xếp vừa tạo thành `part` thì xóa ngay.

Ngăn xếp trên Ví dụ 2: `a, x, x, x, x, y` → đỉnh `"xy"` → bật → `a,x,x,x` → đọc `y` → đỉnh `"xy"` → bật → … → `a`, cuối cùng đọc `b` → `"ab"`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng câu chữ (tìm tay, không dùng `find`)

In [7]:
def remove_brute(s, part):
    m = len(part)
    while True:
        pos = -1
        for i in range(len(s) - m + 1):          # tìm lần xuất hiện trái nhất bằng so sánh tay
            if s[i:i + m] == part:
                pos = i
                break
        if pos == -1:
            return s
        s = s[:pos] + s[pos + m:]                # xóa, tạo chuỗi mới

for s, p, want in examples:
    print(repr(remove_brute(s, p)), "| kỳ vọng", repr(want))

'dab' | kỳ vọng 'dab'
'ab' | kỳ vọng 'ab'


**Độ phức tạp:** mỗi lần xóa: tìm `O(n·m)` + cắt `O(n)`; số lần xóa ≤ `n/m` ⇒ `O(n²)` (tính cả `m`: `O((n/m)·n·m) = O(n²)`). Với `n ≤ 1000` vẫn chạy, nhưng mỗi lần lại **quét từ đầu** — trong khi phần đầu đã chắc chắn sạch.

**Nút thắt:** quét lại phần đã kiểm và tạo chuỗi mới mỗi lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (ngăn xếp):** duy trì ngăn xếp `st` = kết quả đã xử lý xong của tiền tố `s[:i]`. Đọc `s[i]`, đẩy vào; *nếu* `st` kết thúc bằng `part` *thì* bật `len(part)` ký tự. Phép bật có thể làm lộ ra phần cũ, nhưng phần cũ **không** chứa `part` (bất biến) nên chỉ cần kiểm **đỉnh** sau mỗi lần đẩy.

**Bất biến:** sau mỗi bước, `st` **không chứa** `part` ở bất kỳ đâu.

**Insight 2 (KMP):** so hậu tố tốn `O(m)` mỗi bước. Thay vào đó, lưu kèm mỗi phần tử ngăn xếp **trạng thái KMP** sau khi đọc nó. Đẩy ký tự `c`: trạng thái mới = `kmp_step(trạng thái đỉnh, c)`. Nếu trạng thái = `m` ⇒ bật `m` phần tử, trạng thái hiện tại tự động là trạng thái của phần tử đỉnh mới — **không cần tính lại**.

**Vì sao ngăn xếp = "xóa trái nhất"?** Khi đỉnh vừa thành `part`, lần xuất hiện này **kết thúc sớm nhất** trong chuỗi hiện tại (`st` + phần chưa đọc), vì `st` trước đó không chứa `part`. Hai lần xuất hiện cùng độ dài: kết thúc sớm nhất ⇔ **bắt đầu sớm nhất** ⇒ đúng lần trái nhất.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `replace(part, "", 1)` lặp (code ngắn nhất)

In [8]:
class SolutionReplace:
    def removeOccurrences(self, s: str, part: str) -> str:
        while part in s:
            s = s.replace(part, "", 1)
        return s

for s, p, want in examples:
    print(SolutionReplace().removeOccurrences(s, p) == want)

True
True


**Độ phức tạp:** giống Cách 1 về bậc (`O(n²)` worst) nhưng `in`/`replace` chạy bằng C nên nhanh hơn nhiều; với `n ≤ 1000` hoàn toàn đủ. Nhưng phải nhớ **`count = 1`** (Lỗi 1).

### 7.2 Cách 3 — Ngăn xếp + so hậu tố (`O(n · m)`)

In [9]:
def remove_stack_demo(s, part):
    st, m = [], len(part)
    for c in s:
        st.append(c)
        if len(st) >= m and "".join(st[-m:]) == part:
            del st[-m:]
            print(f"đọc {c!r}: đỉnh khớp {part!r} -> bật, stack = {''.join(st)!r}")
        else:
            print(f"đọc {c!r}: stack = {''.join(st)!r}")
    return "".join(st)

print("=>", repr(remove_stack_demo("axxxxyyyyb", "xy")))

đọc 'a': stack = 'a'
đọc 'x': stack = 'ax'
đọc 'x': stack = 'axx'
đọc 'x': stack = 'axxx'
đọc 'x': stack = 'axxxx'
đọc 'y': đỉnh khớp 'xy' -> bật, stack = 'axxx'
đọc 'y': đỉnh khớp 'xy' -> bật, stack = 'axx'
đọc 'y': đỉnh khớp 'xy' -> bật, stack = 'ax'
đọc 'y': đỉnh khớp 'xy' -> bật, stack = 'a'
đọc 'b': stack = 'ab'
=> 'ab'


In [10]:
class SolutionStack:
    def removeOccurrences(self, s: str, part: str) -> str:
        st, m, last = [], len(part), part[-1]
        for c in s:
            st.append(c)
            if c == last and len(st) >= m and "".join(st[-m:]) == part:
                del st[-m:]
        return "".join(st)

for s, p, want in examples:
    print(SolutionStack().removeOccurrences(s, p) == want)

True
True


Mẹo nhỏ: chỉ so hậu tố khi ký tự vừa đẩy **bằng ký tự cuối** của `part`. **Độ phức tạp:** mỗi ký tự đẩy/bật tối đa một lần, mỗi lần so `O(m)` ⇒ `O(n·m)` worst; bộ nhớ `O(n)`.

### 7.3 Cách 4 — Ngăn xếp + KMP (`O(n + m)`)

In [11]:
class SolutionKMP:
    def removeOccurrences(self, s: str, part: str) -> str:
        m = len(part)
        pi = prefix_function(part)
        chars = []            # ký tự trong ngăn xếp
        states = [0]          # states[k] = trạng thái KMP sau k ký tự đầu của ngăn xếp
        for c in s:
            j = states[-1]
            while j and part[j] != c:
                j = pi[j - 1]
            if part[j] == c:
                j += 1
            chars.append(c); states.append(j)
            if j == m:                         # đỉnh vừa thành part
                del chars[-m:]; del states[-m:]
        return "".join(chars)

for s, p, want in examples:
    print(SolutionKMP().removeOccurrences(s, p) == want)

True
True


**Độ phức tạp:** tính `pi` `O(m)`. Mỗi bước đọc ký tự, vòng `while` lùi theo `pi` — phân tích khấu hao của KMP chuẩn dựa trên việc `j` chỉ tăng 1 mỗi bước; ở đây `j` có thể "nhảy về" trạng thái cũ khi bật, nhưng trạng thái đó **đã được tính** chứ không phải tăng dần lại. Trường hợp xấu lý thuyết có thể lớn hơn tuyến tính một chút nếu có nhiều lần lùi lặp lại; muốn chặt chẽ `O(n·σ)` thì dùng **bảng chuyển trạng thái** (automaton) `next[j][c]` dựng trước — xem Bài tập 4. Thực tế với ràng buộc nhỏ cả hai đều tức thời.

### 7.4 Cách 5 — Ngăn xếp + băm lăn (rolling hash)

Một cách khác để kiểm "đỉnh ngăn xếp có bằng `part` không" trong `O(1)`: lưu **băm tiền tố** của ngăn xếp `H[k] = hash(st[:k])`. Băm của `m` ký tự cuối là `H[k] − H[k−m] · B^m (mod M)`. So với băm của `part`; nếu bằng thì **so lại thật** (tránh va chạm băm) rồi bật — bật chỉ cần `del H[-m:]`.

In [12]:
class SolutionHash:
    B, M = 131, (1 << 61) - 1
    def removeOccurrences(self, s: str, part: str) -> str:
        B, M, m = self.B, self.M, len(part)
        hp = 0
        for c in part:
            hp = (hp * B + ord(c)) % M
        Bm = pow(B, m, M)
        st, H = [], [0]
        for c in s:
            st.append(c)
            H.append((H[-1] * B + ord(c)) % M)
            k = len(st)
            if k >= m and (H[k] - H[k - m] * Bm) % M == hp and "".join(st[-m:]) == part:
                del st[-m:]; del H[-m:]
        return "".join(st)

for s_, p_, want in examples:
    print(SolutionHash().removeOccurrences(s_, p_) == want)

True
True


**Độ phức tạp:** mỗi bước `O(1)` cho băm; phần "so lại thật" chỉ chạy khi băm khớp (hầu như chỉ khi khớp thật, mà mỗi lần khớp thật thì bật `m` ký tự — khấu hao vào số ký tự bị xóa) ⇒ `O(n + m)` kỳ vọng. Kỹ thuật băm lăn trên ngăn xếp dùng lại được cho nhiều bài "xóa mẫu lặp".

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 (`part = "abc"`, `pi = [0,0,0]`):

| đọc | trạng thái trước | trạng thái sau | hành động | ngăn xếp |
|---|---|---|---|---|
| d | 0 | 0 | đẩy | `d` |
| a | 0 | 1 | đẩy | `da` |
| a | 1 | 1 | đẩy (lùi rồi khớp lại `a`) | `daa` |
| b | 1 | 2 | đẩy | `daab` |
| c | 2 | **3** | bật 3 | `da` (trạng thái 1) |
| b | 1 | 2 | đẩy | `dab` |
| a | 2 | 1 | đẩy | `daba` |
| a | 1 | 1 | đẩy | `dabaa` |
| b | 1 | 2 | đẩy | `dabaab` |
| c | 2 | **3** | bật | `daba` (trạng thái 1) |
| b | 1 | 2 | đẩy | `dabab` |
| c | 2 | **3** | bật | `dab` |

Kết quả `"dab"`. ✔ Chú ý sau khi bật, trạng thái lấy lại từ `states[-1]` — không tính lại.

In [13]:
def trace_kmp(s, part):
    m = len(part); pi = prefix_function(part)
    chars, states = [], [0]
    for c in s:
        j0 = j = states[-1]
        while j and part[j] != c:
            j = pi[j - 1]
        if part[j] == c:
            j += 1
        chars.append(c); states.append(j)
        act = "đẩy"
        if j == m:
            del chars[-m:]; del states[-m:]; act = f"bật {m}"
        print(f"đọc {c!r}: trạng thái {j0} -> {j} | {act:6} | stack={''.join(chars)!r} (trạng thái đỉnh {states[-1]})")
    return "".join(chars)

trace_kmp("daabcbaabcbc", "abc")

đọc 'd': trạng thái 0 -> 0 | đẩy    | stack='d' (trạng thái đỉnh 0)
đọc 'a': trạng thái 0 -> 1 | đẩy    | stack='da' (trạng thái đỉnh 1)
đọc 'a': trạng thái 1 -> 1 | đẩy    | stack='daa' (trạng thái đỉnh 1)
đọc 'b': trạng thái 1 -> 2 | đẩy    | stack='daab' (trạng thái đỉnh 2)
đọc 'c': trạng thái 2 -> 3 | bật 3  | stack='da' (trạng thái đỉnh 1)
đọc 'b': trạng thái 1 -> 2 | đẩy    | stack='dab' (trạng thái đỉnh 2)
đọc 'a': trạng thái 2 -> 1 | đẩy    | stack='daba' (trạng thái đỉnh 1)
đọc 'a': trạng thái 1 -> 1 | đẩy    | stack='dabaa' (trạng thái đỉnh 1)
đọc 'b': trạng thái 1 -> 2 | đẩy    | stack='dabaab' (trạng thái đỉnh 2)
đọc 'c': trạng thái 2 -> 3 | bật 3  | stack='daba' (trạng thái đỉnh 1)
đọc 'b': trạng thái 1 -> 2 | đẩy    | stack='dabab' (trạng thái đỉnh 2)
đọc 'c': trạng thái 2 -> 3 | bật 3  | stack='dab' (trạng thái đỉnh 2)


'dab'

Trace với `part` **tự chồng lấn** (`"aba"`, `pi = [0,0,1]`) trên `s = "ababa"` — kết quả đúng là `"ba"`:

In [14]:
print("=>", repr(trace_kmp("ababa", "aba")), "| brute:", repr(remove_brute("ababa", "aba")))

đọc 'a': trạng thái 0 -> 1 | đẩy    | stack='a' (trạng thái đỉnh 1)
đọc 'b': trạng thái 1 -> 2 | đẩy    | stack='ab' (trạng thái đỉnh 2)
đọc 'a': trạng thái 2 -> 3 | bật 3  | stack='' (trạng thái đỉnh 0)
đọc 'b': trạng thái 0 -> 0 | đẩy    | stack='b' (trạng thái đỉnh 0)
đọc 'a': trạng thái 0 -> 1 | đẩy    | stack='ba' (trạng thái đỉnh 1)
=> 'ba' | brute: 'ba'


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def removeOccurrences(self, s: str, part: str) -> str:
        m = len(part)
        st = []
        for c in s:
            st.append(c)
            # đỉnh ngăn xếp vừa tạo thành part -> đó chính là lần xuất hiện trái nhất hiện tại
            if len(st) >= m and c == part[-1] and "".join(st[-m:]) == part:
                del st[-m:]
        return "".join(st)

for s, p, want in examples:
    assert Solution().removeOccurrences(s, p) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `T_k` là chuỗi "hiện tại" của quá trình gốc sau `k` lần xóa. Ta chứng minh chuỗi `st + s[i:]` (ngăn xếp nối phần chưa đọc) luôn bằng một `T_k` nào đó, và ngăn xếp không chứa `part`.

**Bất biến:** (A) `st` không chứa `part`; (B) `st + s[i:] = T_k` với `k` = số lần đã bật.

- *Đẩy ký tự `c = s[i]` và không bật:* chuỗi nối không đổi (chỉ dời ranh giới); `st` mới không chứa `part` vì nếu chứa thì lần xuất hiện đó phải dùng `c` (A cũ) ⇒ là hậu tố — mà ta đã kiểm hậu tố không bằng `part`. (A), (B) giữ.
- *Đẩy `c` và đỉnh thành `part`:* lần xuất hiện `P` này kết thúc tại vị trí `|st| − 1` của chuỗi nối `T_k`. Mọi lần xuất hiện khác của `part` trong `T_k` kết thúc **muộn hơn** (vì phần trước đỉnh — `st` cũ — không chứa `part`, nên không có lần nào kết thúc sớm hơn; và cũng không có lần nào khác kết thúc cùng chỗ vì cùng độ dài). Cùng độ dài `m` ⇒ kết thúc muộn hơn ⇔ bắt đầu muộn hơn ⇒ `P` là **lần trái nhất** của `T_k`. Quá trình gốc xóa đúng `P` ⇒ `st` sau khi bật + `s[i+1:]` = `T_{k+1}`. Sau khi bật, `st` là tiền tố của `st` cũ (trước khi đẩy `c`) ⇒ không chứa `part`. (A), (B) giữ.

**Kết thúc:** `i = n`, `st = T_k` và không chứa `part` ⇒ quá trình gốc cũng dừng ở đúng chuỗi này.

**Vì sao cách khác sai:** `replace` toàn bộ hay xóa phải nhất chọn **lần xuất hiện khác** khi có chồng lấn ⇒ kết quả khác (mục 13).

## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 1000`, `m = len(part) ≤ 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng tay | `O(n²)` (tìm tay `O(n·m)` mỗi lần, ≤ `n/m` lần) | `O(n)` |
| 2. `replace(…, 1)` lặp | `O(n²)` (bằng C, nhanh) | `O(n)` |
| 3. Ngăn xếp + hậu tố | `O(n·m)` | `O(n)` |
| 4. Ngăn xếp + KMP | `O(n + m)` thực tế (xem ghi chú 7.3) | `O(n + m)` |
| 5. Ngăn xếp + băm lăn | `O(n + m)` kỳ vọng | `O(n)` |

In [16]:
rng = random.Random(1910)
cases = [
    ("xy lồng nhau n=1000", "x" * 500 + "y" * 500, "xy"),
    ("part dài, ít khớp", "".join(rng.choice("ab") for _ in range(1000)), "ab" * 20),
    ("abc ngẫu nhiên", "".join(rng.choice("abc") for _ in range(1000)), "abc"),
    ("lớn ngoài đề n=2·10^5", "a" * 100000 + "b" * 100000, "ab"),
]
for label, s, p in cases:
    res = []
    for name, f in [("Cách 1", remove_brute), ("Cách 2", SolutionReplace().removeOccurrences),
                    ("Cách 3/Nộp", Solution().removeOccurrences), ("Cách 4 KMP", SolutionKMP().removeOccurrences),
                    ("Cách 5 hash", SolutionHash().removeOccurrences)]:
        if name in ("Cách 1", "Cách 2") and len(s) > 1000:
            continue                                  # O(n²) với 10^5 lần xóa: quá chậm, bỏ qua
        t = time.perf_counter(); r = f(s, p); res.append(r)
        print(f"[{label}] {name}: len kết quả {len(r)} trong {1000*(time.perf_counter()-t):.1f} ms")
    assert len(set(res)) == 1

[xy lồng nhau n=1000] Cách 1: len kết quả 0 trong 5.7 ms
[xy lồng nhau n=1000] Cách 2: len kết quả 0 trong 0.8 ms
[xy lồng nhau n=1000] Cách 3/Nộp: len kết quả 0 trong 0.1 ms
[xy lồng nhau n=1000] Cách 4 KMP: len kết quả 0 trong 0.1 ms
[xy lồng nhau n=1000] Cách 5 hash: len kết quả 0 trong 0.3 ms
[part dài, ít khớp] Cách 1: len kết quả 1000 trong 0.0 ms
[part dài, ít khớp] Cách 2: len kết quả 1000 trong 0.0 ms
[part dài, ít khớp] Cách 3/Nộp: len kết quả 1000 trong 0.2 ms
[part dài, ít khớp] Cách 4 KMP: len kết quả 1000 trong 0.1 ms
[part dài, ít khớp] Cách 5 hash: len kết quả 1000 trong 0.3 ms
[abc ngẫu nhiên] Cách 1: len kết quả 865 trong 1.0 ms
[abc ngẫu nhiên] Cách 2: len kết quả 865 trong 0.1 ms
[abc ngẫu nhiên] Cách 3/Nộp: len kết quả 865 trong 0.1 ms
[abc ngẫu nhiên] Cách 4 KMP: len kết quả 865 trong 0.1 ms
[abc ngẫu nhiên] Cách 5 hash: len kết quả 865 trong 0.3 ms
[lớn ngoài đề n=2·10^5] Cách 3/Nộp: len kết quả 0 trong 17.6 ms
[lớn ngoài đề n=2·10^5] Cách 4 KMP: len kết quả 0 tr

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu mô phỏng)

In [17]:
sols = [Solution(), SolutionReplace(), SolutionStack(), SolutionKMP(), SolutionHash()]
edge = examples + [
    ("aaaa", "aa", ""),
    ("aaa", "aa", "a"),
    ("abc", "d", "abc"),           # không xuất hiện
    ("abc", "abc", ""),            # toàn bộ chuỗi
    ("ab", "abc", "ab"),           # part dài hơn s
    ("ababa", "aba", "ba"),        # chồng lấn: trái nhất
    ("aabababa", "aba", "ba"),
    ("ccctltctlltlb", "ctl", "b"),
]
for s, p, want in edge:
    assert remove_brute(s, p) == want, (s, p, remove_brute(s, p))
    for sol in sols:
        assert sol.removeOccurrences(s, p) == want, (s, p, sol)
    print("OK", repr(s), repr(p), "->", repr(want))

OK 'daabcbaabcbc' 'abc' -> 'dab'
OK 'axxxxyyyyb' 'xy' -> 'ab'
OK 'aaaa' 'aa' -> ''
OK 'aaa' 'aa' -> 'a'
OK 'abc' 'd' -> 'abc'
OK 'abc' 'abc' -> ''
OK 'ab' 'abc' -> 'ab'
OK 'ababa' 'aba' -> 'ba'
OK 'aabababa' 'aba' -> 'ba'
OK 'ccctltctlltlb' 'ctl' -> 'b'


### 12.2 Vét cạn **mọi** chuỗi nhỏ trên bảng chữ `{a, b}`

Mọi `s` dài ≤ 10 và mọi `part` dài ≤ 4 — bao trọn mọi kiểu chồng lấn.

In [18]:
total = 0
for n in range(1, 11):
    for s_t in product("ab", repeat=n):
        s = "".join(s_t)
        for m in range(1, 5):
            for p_t in product("ab", repeat=m):
                p = "".join(p_t)
                want = remove_brute(s, p)
                for sol in sols:
                    assert sol.removeOccurrences(s, p) == want, (s, p)
                total += 1
print("đã kiểm", total, "cặp (s, part)")

đã kiểm 61380 cặp (s, part)


### 12.3 Ngẫu nhiên trên bảng chữ lớn hơn, đúng ràng buộc

In [19]:
rng = random.Random(42)
for _ in range(500):
    alpha = rng.choice(["ab", "abc", "abcd"])
    p = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 5)))
    # chèn part nhiều lần để có nhiều lần xóa liên hoàn
    pieces = []
    for _ in range(rng.randint(1, 40)):
        pieces.append(p if rng.random() < 0.4 else rng.choice(alpha))
    s = "".join(pieces)[:1000] or "a"
    want = remove_brute(s, p)
    for sol in sols:
        assert sol.removeOccurrences(s, p) == want
print("500 case ngẫu nhiên khớp mô phỏng")

500 case ngẫu nhiên khớp mô phỏng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `replace` không có `count = 1` (xóa mọi lần xuất hiện cùng lúc)

In [20]:
def wrong_replace_all(s, part):
    while part in s:
        s = s.replace(part, "")              # SAI: xóa đồng loạt các lần không chồng lấn
    return s

for s, p in [("aabababa", "aba"), ("ababa", "aba"), examples[0][:2]]:
    print(repr(s), repr(p), "sai:", repr(wrong_replace_all(s, p)), "| đúng:", repr(Solution().removeOccurrences(s, p)))

'aabababa' 'aba' sai: 'ab' | đúng: 'ba'
'ababa' 'aba' sai: 'ba' | đúng: 'ba'
'daabcbaabcbc' 'abc' sai: 'dab' | đúng: 'dab'


**Vì sao sai:** khi `part` tự chồng lấn (`"aba"`), xóa đồng loạt chọn các lần xuất hiện **khác** với "lần trái nhất rồi tính lại". Với `"aabababa"`: đúng là `"ba"`, bản sai cho `"ab"`. Trên ví dụ của đề (không chồng lấn) lỗi **không lộ**. **Sửa:** `replace(part, "", 1)`.

### Lỗi 2 — Xóa lần xuất hiện **phải nhất** (`rfind`)

In [21]:
def wrong_rightmost(s, part):
    while part in s:
        i = s.rfind(part)                     # SAI: đề yêu cầu trái nhất
        s = s[:i] + s[i + len(part):]
    return s

print("sai:", repr(wrong_rightmost("ababa", "aba")), "| đúng:", repr(Solution().removeOccurrences("ababa", "aba")))

sai: 'ab' | đúng: 'ba'


**Vì sao sai:** `"ababa"` có hai lần `"aba"` chồng nhau (chỉ số 0 và 2). Xóa trái nhất còn `"ba"`, xóa phải nhất còn `"ab"`.

### Lỗi 3 — Chỉ quét **một lượt**, không xử lý lần xuất hiện mới sinh ra

In [22]:
def wrong_one_pass(s, part):
    out, i, m = [], 0, len(part)
    while i < len(s):
        if s.startswith(part, i):
            i += m                            # SAI: bỏ qua part nhưng không kiểm chỗ nối
        else:
            out.append(s[i]); i += 1
    return "".join(out)

for s, p, want in examples:
    print(repr(s), "sai:", repr(wrong_one_pass(s, p)), "| đúng:", repr(want))

'daabcbaabcbc' sai: 'dababc' | đúng: 'dab'
'axxxxyyyyb' sai: 'axxxyyyb' | đúng: 'ab'


**Vì sao sai:** xóa xong, phần trái và phải dính lại có thể tạo `part` mới (Ví dụ 2 là trường hợp điển hình). **Sửa:** ngăn xếp kiểm đỉnh **sau mỗi lần đẩy** (kể cả ngay sau khi bật).

### Lỗi 4 — Ngăn xếp nối cả chuỗi mỗi bước để kiểm (`"".join(st).endswith`)

In [23]:
def slow_join_all(s, part):
    st = []
    for c in s:
        st.append(c)
        if "".join(st).endswith(part):        # đúng nhưng O(n) mỗi bước -> O(n²)
            del st[-len(part):]
    return "".join(st)

big_s = "ab" * 6000                          # không có lần xuất hiện nào -> ngăn xếp dài dần
t = time.perf_counter(); r1 = slow_join_all(big_s, "abc"); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().removeOccurrences(big_s, "abc"); t2 = time.perf_counter() - t
assert r1 == r2
print(f"join cả stack: {1000*t1:.0f} ms | chỉ so m ký tự cuối: {1000*t2:.0f} ms (n = 1.2·10^4, ngoài ràng buộc)")

join cả stack: 228 ms | chỉ so m ký tự cuối: 1 ms (n = 1.2·10^4, ngoài ràng buộc)


**Vì sao chưa tốt:** kết quả đúng, nhưng mỗi bước dựng lại cả chuỗi `O(n)` ⇒ `O(n²)`; với `n ≤ 1000` vẫn qua, nhưng là thói quen xấu. **Sửa:** chỉ so `m` phần tử cuối `st[-m:]`.

### Lỗi 5 — KMP trên ngăn xếp nhưng **reset trạng thái về 0** sau khi bật

In [24]:
def wrong_kmp_reset(s, part):
    m = len(part); pi = prefix_function(part)
    st, j = [], 0
    for c in s:
        while j and part[j] != c:
            j = pi[j - 1]
        if part[j] == c:
            j += 1
        st.append(c)
        if j == m:
            del st[-m:]
            j = 0                              # SAI: phải là trạng thái của đỉnh mới
    return "".join(st)

for s, p in [examples[0][:2], examples[1][:2]]:
    print(repr(s), "sai:", repr(wrong_kmp_reset(s, p)), "| đúng:", repr(Solution().removeOccurrences(s, p)))

'daabcbaabcbc' sai: 'dababc' | đúng: 'dab'
'axxxxyyyyb' sai: 'axxxyyyb' | đúng: 'ab'


**Vì sao sai:** sau khi bật, phần còn lại của ngăn xếp có thể đang "khớp dở" `part` (Ví dụ 2: sau khi bật `"xy"`, đỉnh là `"x"` — đã khớp 1 ký tự). Reset về 0 làm mất thông tin đó ⇒ bỏ sót lần xuất hiện mới. **Sửa:** lưu trạng thái cho **từng** vị trí ngăn xếp (Cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xóa lặp đi lặp lại một mẫu, xóa xong có thể sinh mẫu mới"** ⇒ ngăn xếp, kiểm đỉnh sau mỗi lần đẩy (giống 1047 *Remove All Adjacent Duplicates*, 1209, 2696, 1717).
- Mẫu nhiều ký tự + cần tuyến tính ⇒ lưu **trạng thái KMP** (hoặc trạng thái automaton) theo từng phần tử ngăn xếp.
- `str.replace(x, y)` thay **tất cả** — đọc kỹ đề có yêu cầu "trái nhất mỗi lần" không.
- Thử ngay với `part` **tự chồng lấn** (`"aba"`, `"aa"`) để phát hiện cách làm sai.
- Ràng buộc nhỏ (`≤ 1000`) ⇒ `while part in s: s = s.replace(part, "", 1)` là lời giải ngắn, đủ nhanh.

## 15. Biến thể của bài

1. **Xóa lần xuất hiện phải nhất** ⇒ ngăn xếp quét **từ phải sang trái** với `part` đảo ngược (Bài tập 5).
2. **Nhiều mẫu cần xóa** ⇒ Aho–Corasick + ngăn xếp trạng thái.
3. **LeetCode 1047 / 1209** — xóa cặp/k ký tự liền kề giống nhau (ngăn xếp `(ký tự, số lượng)`).
4. **LeetCode 1717 / 2216** — xóa `"ab"`/`"ba"` có điểm — ngăn xếp + tham lam.
5. **Đếm số lần xóa** hoặc **vị trí gốc** của các lần xóa (Bài tập 3).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng tay | tìm trái nhất, cắt | `O(n²)` | `O(n)` | chuẩn đối chiếu |
| 2. `replace(…,1)` lặp | như 1, dùng hàm C | `O(n²)` | `O(n)` | ràng buộc nhỏ, code 2 dòng |
| 3. Ngăn xếp + hậu tố | xóa ngay khi đỉnh khớp | `O(n·m)` | `O(n)` | **nộp bài** — rõ, nhanh |
| 4. Ngăn xếp + KMP | trạng thái theo từng vị trí | `~O(n + m)` | `O(n+m)` | `n`, `m` lớn |
| 5. Ngăn xếp + băm lăn | băm tiền tố của ngăn xếp | `O(n + m)` kỳ vọng | `O(n)` | thay KMP, dễ nhớ |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `("aaaa","aa")`, `("abc","d")`, `("aaabbb","ab")`?

**Bài 2 (tính tay):** Trace ngăn xếp (Cách 3) trên `s = "aabababa"`, `part = "aba"`.

**Bài 3 (điền TODO):** Viết `count_removals(s, part)` trả về **số lần xóa** đã thực hiện.

**Bài 4 (tự viết lại):** Dựng **automaton KMP** `nxt[j][c]` (bảng chuyển trạng thái cho mọi `j ∈ [0, m)` và mọi chữ cái) rồi viết lại Cách 4 sao cho mỗi ký tự xử lý đúng `O(1)`.

**Bài 5 (mini challenge):** Viết `remove_rightmost(s, part)` — quá trình xóa lần xuất hiện **phải nhất** cho tới khi hết — trong `O(n·m)` (không mô phỏng cắt chuỗi), đối chiếu với mô phỏng.

In [25]:
def count_removals(s, part):
    # TODO
    pass

def build_automaton(part):
    # TODO: trả về list m dict/list chuyển trạng thái
    pass

def remove_rightmost(s, part):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `"aaaa"` có 2 lần `"aa"` | `"abc"` không chứa `"d"` | `"aaabbb"` → `""` (xóa liên hoàn) |
| 2 | Đẩy `a,a,b,a` → đỉnh `aba` → bật | còn `a`, đọc tiếp `b,a` … | Kết quả `"ba"` |
| 3 | Mỗi lần bật là một lần xóa | Đếm số lần `del` | Hoặc `(len(s) − len(kết quả)) // len(part)` |
| 4 | `nxt[j][c]` = `j+1` nếu `part[j] == c` | Ngược lại `nxt[pi[j−1]][c]` (với `j > 0`), hoặc 0 | Thêm trạng thái `m` không cần vì bật ngay |
| 5 | Xóa phải nhất = xóa trái nhất trên chuỗi **đảo ngược** | Đảo cả `s` và `part` | Đảo kết quả lại |

## 19. Đáp án tham khảo

In [26]:
r1 = [Solution().removeOccurrences(*a) for a in [("aaaa", "aa"), ("abc", "d"), ("aaabbb", "ab")]]
print("Bài 1:", r1); assert r1 == ["", "abc", ""]
print("Bài 2:"); assert remove_stack_demo("aabababa", "aba") == "ba"

Bài 1: ['', 'abc', '']
Bài 2:
đọc 'a': stack = 'a'
đọc 'a': stack = 'aa'
đọc 'b': stack = 'aab'
đọc 'a': đỉnh khớp 'aba' -> bật, stack = 'a'
đọc 'b': stack = 'ab'
đọc 'a': đỉnh khớp 'aba' -> bật, stack = ''
đọc 'b': stack = 'b'
đọc 'a': stack = 'ba'


In [27]:
def count_removals(s, part):
    st, m, k = [], len(part), 0
    for c in s:
        st.append(c)
        if len(st) >= m and "".join(st[-m:]) == part:
            del st[-m:]; k += 1
    return k

def build_automaton(part):
    m = len(part); pi = prefix_function(part)
    nxt = [[0] * 26 for _ in range(m)]
    for j in range(m):
        for ci in range(26):
            c = chr(97 + ci)
            if part[j] == c:
                nxt[j][ci] = j + 1
            elif j > 0:
                nxt[j][ci] = nxt[pi[j - 1]][ci]
    return nxt

def remove_automaton(s, part):
    m = len(part); nxt = build_automaton(part)
    chars, states = [], [0]
    for c in s:
        j = nxt[states[-1]][ord(c) - 97]
        chars.append(c); states.append(j)
        if j == m:
            del chars[-m:]; del states[-m:]
    return "".join(chars)

def count_removals_brute(s, part):
    k = 0
    while part in s:
        s = s.replace(part, "", 1); k += 1
    return k

rng = random.Random(5)
for _ in range(800):
    p = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    s = "".join(rng.choice(["a", "b", p]) for _ in range(rng.randint(1, 20)))
    assert count_removals(s, p) == count_removals_brute(s, p)
    assert remove_automaton(s, p) == remove_brute(s, p)
print("Bài 3, 4: OK", count_removals("daabcbaabcbc", "abc"), remove_automaton("axxxxyyyyb", "xy"))

Bài 3, 4: OK 3 ab


In [28]:
def remove_rightmost(s, part):
    return Solution().removeOccurrences(s[::-1], part[::-1])[::-1]

def remove_rightmost_sim(s, part):
    while part in s:
        i = s.rfind(part)
        s = s[:i] + s[i + len(part):]
    return s

for n in range(1, 10):
    for s_t in product("ab", repeat=n):
        s = "".join(s_t)
        for p in ["a", "ab", "aba", "aab", "bab", "abab"]:
            assert remove_rightmost(s, p) == remove_rightmost_sim(s, p), (s, p)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", remove_rightmost("ababa", "aba"))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK ab


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
st, m = [], len(part)
for c in s:
    st.append(c)
    if len(st) >= m and "".join(st[-m:]) == part:   # đỉnh vừa thành part
        del st[-m:]                                   # = xóa lần trái nhất hiện tại
return "".join(st)

# ngắn (ràng buộc nhỏ):
while part in s: s = s.replace(part, "", 1)
```

**Checklist:**
- [ ] Xóa **trái nhất**, mỗi lần **một** lần xuất hiện (`count = 1`)?
- [ ] Xử lý lần xuất hiện **mới sinh ra** ở chỗ nối (kiểm đỉnh sau mỗi lần đẩy)?
- [ ] Đã thử `part` tự chồng lấn (`"aba"` trên `"ababa"` → `"ba"`)?
- [ ] Chỉ so `m` ký tự cuối, không nối cả ngăn xếp?
- [ ] Nếu dùng KMP: lưu trạng thái cho từng vị trí ngăn xếp, không reset về 0.